### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model=init_chat_model("groq:qwen/qwen3-32b")
response=model.invoke("whats up")
response

AIMessage(content='<think>\nOkay, the user asked "whats up". I need to respond appropriately. Let me think about the right way to reply.\n\nFirst, they might just be greeting me or checking if I\'m online. Since I\'m an AI, I should acknowledge their message warmly. Maybe say something like "Hello!" or "Hi!" to be friendly.\n\nThey could be looking for a conversation or need help with something specific. I should offer assistance in case they have questions or need support. But since they didn\'t mention anything specific, I don\'t want to assume. Just keep it open-ended.\n\nI should keep the tone casual and approachable. Avoid any formal language since the user used an informal greeting. Maybe add an emoji to keep it light and friendly.\n\nAlso, make sure the response is concise but inviting. Let them know I\'m here to help if they need anything. Something like, "Hello! Just checking in. How can I assist you today?" That sounds good. Wait, maybe "Hi there! Just a friendly hello. How c

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"The weather at {location} is sunny"

model_with_tools=model.bind_tools([get_weather])

In [3]:
response=model_with_tools.invoke("What's the weather like in Boston?")
response
print(response)
for tool_call in response.tool_calls:
    #view tool calls made by the model
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'q77p22nr3', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.147814872, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.006719772, 'prompt_tokens_details': None, 'queue_time': 0.055747832, 'total_time': 0.154534644}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6cd5-48b7-76e2-b

### Tools Execution Loops

In [7]:
messages=[{"role":"user","content":"What's the weaher in Boston?"}]
ai_msg=model_with_tools.invoke(messages)
print(messages)
messages.append(ai_msg)
print(messages)


for tool_call in ai_msg.tool_calls:
    print(tool_call)
    tool_result=get_weather.invoke(tool_call)
    print(tool_result)
    messages.append(tool_result)
    print(messages)

final_response=model_with_tools.invoke(messages)
print(final_response.text)



[{'role': 'user', 'content': "What's the weaher in Boston?"}]
[{'role': 'user', 'content': "What's the weaher in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. The user spelled "weaher" instead of "weather", but I understand they want the current weather. I need to call get_weather with the location set to Boston. No other parameters are needed. Let me make sure to format the tool call correctly in JSON inside the XML tags.\n', 'tool_calls': [{'id': 'hf6yp9kkx', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 155, 'total_tokens': 269, 'completion_time': 0.191782906, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.006512152, 'prompt_tokens_details': None

# IMPORTANT NOTES

### Important Concept
 > bind_tools() does NOT execute the function automatically in Python.
 It only gives the model the ABILITY to request tool usage.

### STEP-1
``` Python
response = model_with_tools.invoke(
    "What's the weather like in Boston?"
)
```

or 

``` Python
ai_msg = model_with_tools.invoke(messages)
```
#
❌ Python function is NOT executed yet. The model ONLY says:
> "I want to call this tool."

#
That’s why you got:

```Python
tool_calls=[
   {
      'name': 'get_weather',
      'args': {'location': 'Boston'}
   }
]
```

This is NOT the tool result.It is only: **LLM REQUESTING TOOL EXECUTION**

### VERY IMP. CONCEPT
The LLM says:
```
"Hey system, please run:
get_weather(location='Boston')"
```
But the LLM itself CANNOT execute Python code.

So YOU (or LangChain agent framework) must execute it.

***THAT is why this loop exists***
``` Python
for tool_call in ai_msg.tool_calls:
```
Because now you are reading:

- which tool to call
- what arguments to pass

### THIS LINE ACTUALLY RUNS THE FUNCTION
``` Python
tool_result = get_weather.invoke(tool_call)
```
THIS is the real execution.

### Then this gets returned

``` Python
"The weather at Boston is sunny"
```

### WHY append tool_result to messages?
This is the MOST IMPORTANT PART.

The LLM still doesn't know the tool output.

Remember:

- LLM requested tool
- external system executed tool

Now we must give result BACK to LLM.

So:

``` Python
messages.append(tool_result)
```

adds:

```
Tool said:
"The weather at Boston is sunny"
```

to conversation history.

### THEN THIS HAPPENS
``` Python
final_response = model_with_tools.invoke(messages)
```

Now model sees:

```
User:
What's weather in Boston?

Assistant:
CALLING TOOL get_weather()

Tool:
The weather at Boston is sunny
```
NOW the model can generate final natural-language response:

```
The weather in Boston is currently sunny ☀️
```
### SO THE FLOW IS:
```
User Question
      ↓
LLM decides tool needed
      ↓
LLM outputs tool_call
      ↓
Python executes tool
      ↓
Tool result appended to chat
      ↓
LLM reads tool result
      ↓
Final human-friendly answer
```

### Then why do agents feel automatic?
Because LangChain Agents internally do ALL these steps automatically:
```
LLM tool request
→ execute tool
→ append tool result
→ call LLM again
```

behind the scenes.

So when using:
``` Python
agent.invoke()
```
everything feels automatic.

But here you are manually implementing agent loop.

Which is GOOD for learning.


### "Why get_weather.invoke instead of get_weather()?"

Because in LangChain: >get_weather
is not a normal Python function anymore. It becomes a LangChain Tool object.

LangChain tools support: **.invoke()**

Normal Python:
``` Python 
get_weather("Boston")
```

LangChain Tool:
``` Python
get_weather.invoke({
    "location":"Boston"
})
```

because tools work with structured inputs.

### Why LangChain uses invoke()

Because EVERYTHING in LangChain uses common interface:

- models
- chains
- tools
- runnables

all support:
```
.invoke()
.stream()
.batch()
```
Unified architecture.


### Important 
``` Python
response = model_with_tools.invoke(
    "What's the weather like in Boston?"
)
```

This is shorthand.

Internally LangChain converts it to:

```
[
    {
        "role": "user",
        "content": "What's the weather like in Boston?"
    }
]
```
### WHY DO WE NEED MESSAGE FORMAT?

Because agents/tool calling are conversational.

After tool calling, conversation becomes:

```
User → Assistant(tool call) → Tool → Assistant(final response)
```
A plain string cannot represent this complex history.

So for advanced workflows:

tools
memory
multi-turn chats
agents

you use message lists.
